# 00 · Entendimiento del negocio

**Fase CRISP-DM:** 1. Entendimiento del negocio

**Objetivo:** definir el problema, quién lo tiene, cómo se mide el éxito y qué entra en el alcance, **antes** de tocar los datos.

**Doble propósito del proyecto:**

- **Proyecto del curso:** ciclo CRISP-DM completo (notebooks 00 a 15), con datos sintéticos.
- **Producto Puzzle:** el mismo código pasa a producción cuando lleguen los datos reales (`docs/propuesta.md`).

> Este notebook solo narra. La lógica vive en `src/puzzle_recsys` y los parámetros en `config/`.

## Configuración

In [1]:
import pandas as pd

from puzzle_recsys.config import load_paths, load_yaml

paths, mcfg = load_paths(), load_yaml("models")

## 1. Problema y usuarios

**Puzzle** es una app para pedir en restaurantes. Un comensal registrado entra a un restaurante, muchas veces **por primera vez**, abre el menú y tiene que elegir entre decenas de platillos que no conoce.

**El problema:** el menú está ordenado igual para todos. Quien detesta el picante ve primero los chiles en nogada; quien siempre pide bebida sin alcohol ve primero la carta de mezcales. Elegir cuesta tiempo y a veces termina en un platillo que no gusta.

**La propuesta:** dos listas personalizadas al abrir el menú.

| Lista | Para quién | Qué ordena |
|---|---|---|
| **Para ti** | cada comensal registrado | platillos individuales, por tipo (entrada, fuerte, bebida, postre, guarnición) |
| **Para la mesa** | el grupo sentado junto | platillos para compartir, según los gustos combinados de la mesa |

**La idea clave:** en un restaurante nuevo, el comensal nunca interactuó con sus platillos. Lo que se transfiere entre restaurantes son los **atributos** (cocina, proteína, ingredientes, picante, precio) y el **platillo canónico**: "Spaghetti alla carbonara" y "Pasta carbonara de la casa" son el mismo `carbonara`. Los gustos se aprenden en ese espacio, no en el de los identificadores de cada restaurante.

**Usuarios y partes interesadas:**

- **Comensal:** elige más rápido y mejor. Sus alergias se respetan siempre (filtro duro, no una preferencia).
- **Restaurante:** sus platillos llegan a quien probablemente los disfrute, no solo los más populares.
- **Puzzle:** más pedidos satisfechos y más retención.

**Restricción del caso:** Puzzle todavía no guarda la identidad persistente del comensal, los atributos de los platillos ni el historial de precios. Por eso el proyecto se desarrolla con **datos sintéticos** anclados en fuentes públicas (INEGI, DENUE, ENIGH, Censo 2020) y propone los cambios de base de datos necesarios (`docs/cambios-base-de-datos.md`).

## 2. KPIs y criterio de éxito

**Métrica principal:** **NDCG@3**, que premia poner lo que la persona va a pedir **en los primeros lugares** de la lista. Se usa el top 3 porque cada consulta tiene pocos candidatos (mediana de 5, notebook 06): un top 10 incluiría casi todo el menú y no distinguiría modelos.

**Métricas de apoyo:**

- **HitRate@3:** ¿lo que pidió quedó entre los 3 primeros? Es la más fácil de explicar al negocio.
- **Recall@k y MAP@k:** cobertura de los positivos y precisión promedio.
- **Cobertura del catálogo:** qué parte de los platillos llega alguna vez al top 3. Vigila el **sesgo de popularidad** (recomendar siempre lo mismo).
- **Por segmento:** tipo de plato, historia del comensal y primera visita al restaurante.

**Criterio de éxito**, fijado **antes** de mirar la prueba (`config/models.yaml`):

1. El modelo debe superar a la **popularidad** (lo más pedido), que es lo que Puzzle podría mostrar hoy sin aprendizaje.
2. La mejora debe ser **relevante**: al menos **0.01 de NDCG@3**. Menos que eso no se nota en una lista de 3 a 5 platillos y no paga mantener un modelo más complejo.
3. La mejora debe ser **real**: intervalo de confianza por comensal por encima de ese valor, con una prueba con **potencia** suficiente, verificada por simulación (notebook 13).

In [2]:
pd.Series({
    "métrica principal": "NDCG@3",
    "k evaluados": mcfg["evaluation"]["k"],
    "mejora mínima relevante (NDCG@3)": mcfg["evaluation"]["min_relevant_improvement"],
    "remuestreos bootstrap": mcfg["evaluation"]["bootstrap_iterations"],
    "permutaciones": mcfg["evaluation"]["permutation_iterations"],
    "partición (meses)": f"train {mcfg['split']['train']}, valid {mcfg['split']['valid']}, test {mcfg['split']['test']}",
}, name="valor").to_frame()

,valor
métrica principal,NDCG@3
k evaluados,"[3, 5]"
mejora mínima relevante (NDCG@3),0.01
remuestreos bootstrap,2000
permutaciones,5000
partición (meses),"train [1, 9], valid [10, 10], test [11, 12]"


## 3. Alcance

| Núcleo (entra siempre) | Si alcanza el tiempo | Fase 2 |
|---|---|---|
| Simulación completa, EDA, calidad, particiones, "para ti" con 4 modelos (popularidad, contenido, logística, LightGBM), Optuna, SHAP, evaluación por segmentos | Ranker completo de "para la mesa", análisis de potencia, calibración por hipercubo latino | Recomendar restaurantes, modelos profundos (NCF, RNN), corrección del sesgo de posición, efecto de los acompañantes en "para ti" |

**Resultado:** el núcleo y todo lo de "si alcanza" quedaron implementados (notebooks 01 a 15).

**Decisiones de alcance tomadas en el camino** (para no complicar sin necesidad):

- **No** se modela quién comió cada platillo compartido: solo que la mesa lo compartió.
- Los menús reales extraídos de la web quedan como **anexo**, no se integran a la simulación.
- El modelo se entrena con una **muestra del 30 %** de las cuentas registradas (notebook 10), por la memoria disponible.
- **No** se simulan registros escalonados a lo largo del año. El arranque en frío se evalúa de forma **contrafactual** (notebook 13).

## 4. Riesgos y supuestos

**Supuestos:** todo lo que no sale de datos reales está declarado en `docs/supuestos.md` (S1 a S17), con su justificación y el notebook donde se usa. Los más importantes:

- Los gustos latentes y la forma de elegir son una **verdad conocida** (S4): sirve para validar el método, no el rendimiento real.
- El efecto de la **hora** (S16) y del **gusto por compartir** de la mesa (S17) son supuestos basados en hábitos conocidos de la CDMX, no estimaciones. Son lo primero a recalibrar con datos reales.

**Riesgos:**

| Riesgo | Mitigación |
|---|---|
| El modelo aprende la simulación y no la realidad | validar el método aquí y medir el rendimiento real con una prueba A/B en Puzzle (notebook 15) |
| Fuga de información (usar el futuro para predecir el pasado) | todo se calcula "a la fecha"; partición temporal; pruebas de fuga en `tests/` |
| Arranque en frío al lanzar la app (nadie tiene historia) | onboarding como prior; evaluación contrafactual de persona y restaurante nuevos |
| Sesgo de popularidad | métrica de cobertura del catálogo |
| Alergias | filtro **duro** antes del modelo; el modelo nunca decide sobre seguridad alimentaria |
| Datos personales (alergias = datos sensibles, LFPDPPP) | solo comensales registrados con consentimiento de personalización |

## Conclusiones

- **Problema:** ordenar el menú para cada comensal ("para ti") y para cada mesa ("para la mesa"), sobre todo en restaurantes que la persona no conoce.
- **Éxito:** superar a la popularidad por al menos **0.01 de NDCG@3**, con significancia por comensal y potencia verificada.
- **Pasa al notebook 01:** construir el mundo simulado con fuentes públicas reales.